In [1]:
   import pandas as pd
   df = pd.read_json("companies.jsonl", lines=True)
   df.head()

,website,operational_name,year_founded,address,employee_count,revenue,primary_naics,description,business_model,target_markets,core_offerings,is_public,secondary_naics
0,rompetrol.ro,Rompetrol,1979.0,"{'country_code': 'ro', 'latitude': 44.4775537,...",NaN,1.349824e+10,"{'code': '324110', 'label': 'Petroleum Refiner...",Rompetrol is a Romanian company specialized in...,"[Wholesale, Manufacturing, Business-to-Busines...","[Energy, Industrial, Transportation]","[Fuel Product Distribution, Quality and Safety...",False,None
1,unilever.ro,Unilever,NaN,"{'country_code': 'ro', 'latitude': 44.4761769,...",NaN,1.280207e+10,"{'code': '424690', 'label': 'Other Chemical an...","Unilever South Central Europe SA, DBA Unilever...","[Retail, Business-to-Business, Business-to-Con...",[Consumer Goods],"[Consumer Goods Distribution and Retail, Ingre...",False,None
2,rompetrol.com,Rompetrol,1974.0,"{'country_code': 'ro', 'latitude': 44.47775356...",7000.0,1.266496e+10,"{'code': '324110', 'label': 'Petroleum Refiner...",Rompetrol is a Romanian energy company engaged...,"[Wholesale, Manufacturing, Business-to-Busines...","[Energy, Petrochemicals]","[Sustainable Energy Solutions, Fuel and Energy...",False,None
3,cbre.ro,CBRE Romania,2008.0,"{'country_code': 'ro', 'latitude': 44.4540414,...",92.0,9.568739e+09,"{'code': '531210', 'label': 'Offices of Real E...",CBRE Romania is a Romanian company engaged in ...,"[Service Provider, Business-to-Business, Enter...","[Hospitality, Retail, Industrial, Real Estate,...","[Data Center Real Estate Services, Real Estate...",False,None
4,timacagro.com,TIMAC AGRO,NaN,"{'country_code': 'ro', 'latitude': 44.4792186,...",65.0,8.907855e+09,"{'code': '325311', 'label': 'Nitrogenous Ferti...",TIMAC AGRO S.R.L. is a French agricultural com...,"[Business-to-Business, Enterprise, Manufacturi...",[Agriculture],"[Agricultural Product Retail, Fertilizer Manuf...",False,None


In [2]:
df.shape

(477, 13)

477 de firme

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 477 entries, 0 to 476
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   website           439 non-null    object 
 1   operational_name  475 non-null    object 
 2   year_founded      346 non-null    float64
 3   address           477 non-null    object 
 4   employee_count    289 non-null    float64
 5   revenue           384 non-null    float64
 6   primary_naics     477 non-null    object 
 7   description       477 non-null    object 
 8   business_model    477 non-null    object 
 9   target_markets    477 non-null    object 
 10  core_offerings    477 non-null    object 
 11  is_public         477 non-null    bool   
 12  secondary_naics   11 non-null     object 
dtypes: bool(1), float64(3), object(9)
memory usage: 45.3+ KB


obervam ca majoritatea conditiilor sunt completate (address, primary_naics, description, business_model, target_markets, core_offerings, is_public), avem cateva care au lipsuri mici sau medii (restul fara cele care au 477/477 si fara secondary_natics)
year_found e de tip float, daca nu lipsea nimic era de tip int (numeric) => numarul de valori NaN se deduce din diferenta 477-346=131

In [4]:
(df.isna().mean() * 100).round(1).sort_values(ascending=False)

secondary_naics     97.7
employee_count      39.4
year_founded        27.5
revenue             19.5
website              8.0
operational_name     0.4
address              0.0
primary_naics        0.0
description          0.0
business_model       0.0
target_markets       0.0
core_offerings       0.0
is_public            0.0
dtype: float64

am sortat valorile precedente de la cea mai mare la cea mai mica si le am facut sub forma de procent

employee_count este la 39%, daca l elimin direct pierd aproape 4 din 10 firme

secondary_naics este la 98%, adica 98% din firme nu o au, nu ne putem baza pe el ,dar daca e nevoie o sa l folosim ca bonus

ca "piloni" pe care ne putem baza avem: description, business_model, target_markets, core_offerings, address, primary_naics si is_public

daca unei firme ii lipseste un camp cerut de cerere dar indeplineste alte x criterii, il mutam in partea de jos a clasamentului (momentan)

In [5]:
(df.business_model.apply(len)==0).sum() 

np.int64(0)

In [6]:
(df.target_markets.apply(len)==0).sum()

np.int64(0)

In [7]:
(df.core_offerings.apply(len)==0).sum()

np.int64(0)

cele 3 coloane (business_model, target_markets si core_offerings) nu sunt goale 

In [8]:
lungimi=df.description.str.len()
lungimi

0      492
1      449
2      582
3      785
4      507
      ... 
472    492
473    575
474    459
475    615
476    473
Name: description, Length: 477, dtype: int64

In [9]:
lungimi.describe()

count    477.000000
mean     556.075472
std      140.501273
min      121.000000
25%      469.000000
50%      558.000000
75%      648.000000
max      913.000000
Name: description, dtype: float64

In [10]:
(lungimi < 350).sum()

np.int64(28)

In [11]:
lungimi.sort_values()

382    121
416    134
171    167
425    167
166    177
      ... 
365    885
39     887
281    905
294    912
295    913
Name: description, Length: 477, dtype: int64

In [12]:
df.loc[166,'description']

'Siemens Energy Oy, DBA Siemens Energy, is a Finnish company operating in the energy equipment and solutions sector, providing products and services related to energy technology.'

observam ca minimul lungimii unei descrieri este de 121 iar maximul de 913, majoritatea fiind pe la 558, am încercat inițial un prag de 350

dar cum a fost si in exemplul de mai sus df.loc[166,'description'], descrierea a fost buna desi aceasta avea dublu sub numarul impus de mine de caractere

adica nu eliminam nici descrierile scurte doar ca o sa le dam un double check dupa niste cuvinte cheie

exemple de cereri care depind de campurile cu lipsuri (din lista de 12):

cererea 2 (Public software companies with more than 1,000 employees) depinde de employee_count

cererea 5 (Construction companies in the United States with revenue over $50 million) depinde de revenue

cererea 8 (Clean energy startups founded after 2018 with fewer than 200 employees) depinde de year_founded si de employee_count

concluzia este ca lipsurile din date afecteaza doar 3 din cele 12 cereri, pentru celelalte 9, campurile lipsa nu conteaza ,dar faptul că o cerere nu e afectată de lipsuri nu înseamnă că e ușoară. Cea mai riscanta fiind cererea 8 ,deoarece are 2 lipsuri

In [13]:
df.address[0]

"{'country_code': 'ro', 'latitude': 44.4775537, 'longitude': 26.0713416, 'region_name': 'Bucharest', 'town': 'Bucharest'}"

In [14]:
df.address[5]

"{'country_code': 'ro', 'latitude': 44.490331, 'longitude': 26.0556503, 'region_name': 'Bucharest', 'town': 'Bucharest'}"

In [15]:
type(df.address[0])

str

In [16]:
type(df.address[5])

str

In [17]:
df.primary_naics[0]

"{'code': '324110', 'label': 'Petroleum Refineries'}"

In [18]:
type(df.primary_naics[0])

str

In [19]:
type(df.primary_naics[5])

str

observam ca datele (pentru cele 2 exemple [0] si [5]) din campurile address si primary_naics sunt str (text), DAR nu putem exclude faptul ca pot fi si list sau dict

in exemplul nostru avem adresa "address": "Munich, Germany" iar noi avem ro care este prescurtarea tarii



In [20]:
df.address.apply(type).value_counts()

address
<class 'dict'>    413
<class 'str'>      64
Name: count, dtype: int64

asa cum am presupus nu avem doar str avem si dict la o mare moajoritate din firme

In [21]:
df.primary_naics.apply(type).value_counts()

primary_naics
<class 'dict'>    413
<class 'str'>      64
Name: count, dtype: int64

la fel si in cazul campului primary_naics

In [22]:
import ast
adresa = ast.literal_eval(df.address[0])
type(adresa)

dict

In [23]:
adresa["country_code"]

'ro'

am observat ca python nu poate sa scoata un anumit cuvant dintr-un str, deoarece el este un str care arata ca un dict. dupa l am facaut pe acesta dict

apoi am citit country_code din acesta si ne da country_code-ul "ro", putem face asa pentru toata lista de firme daca aceasta e str nu dict

In [24]:
def to_dict(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

In [25]:
type(to_dict(df.address[0])) 
type(to_dict(df.address[100]))

dict

In [26]:
print(to_dict(df.address[0]))
print(to_dict(df.address[100]))

{'country_code': 'ro', 'latitude': 44.4775537, 'longitude': 26.0713416, 'region_name': 'Bucharest', 'town': 'Bucharest'}
{'country_code': 'es', 'latitude': 41.5204464, 'longitude': 1.9024126, 'region_name': 'Catalonia', 'town': 'Abrera'}


In [27]:
df["address"] = df.address.apply(to_dict)

In [28]:
df.address.apply(type).value_counts()

address
<class 'dict'>    477
Name: count, dtype: int64

In [29]:
df["primary_naics"] = df.primary_naics.apply(to_dict)

In [30]:
df.primary_naics.apply(type).value_counts()

primary_naics
<class 'dict'>    477
Name: count, dtype: int64

am definit o functie (dict) cu care am parcurs pe rand ambele coloane (address si primary_naics) am verificat tipul textului (str sau dict). daca era dict l-am returnat asa cum era si daca era str l-am transformat in dict cu functia "ast.literal_eval(x)"

dupa am reluat value_counts pentru ambele si de la 413 dict si 64 str am ajuns la 477 dict

In [31]:
df["country"]=df.address.str.get("country_code")
df.country

0      ro
1      ro
2      ro
3      ro
4      ro
       ..
472    ch
473    ch
474    ch
475    ch
476    ch
Name: country, Length: 477, dtype: object

In [32]:
df.country.value_counts()

country
us    86
ch    43
fr    40
cn    35
se    31
no    27
gb    27
ro    26
dk    26
es    19
fi    13
de    12
in    11
nl     8
au     8
ca     8
it     5
ie     4
kr     4
sg     4
is     4
pl     3
lu     3
be     3
jp     2
id     2
gr     2
nz     2
br     2
pt     2
at     2
ru     2
kw     1
ua     1
lt     1
vn     1
tr     1
hr     1
ar     1
eg     1
cl     1
tw     1
hk     1
Name: count, dtype: int64

avem 43 de tari diferite, cele mai multe fiind din sua (us), elvetia (ch) si franta (fr)

din firmele care apar in cereri avem un total de 207 firme (ro = 26, fr = 40, ch = 43, de = 12, us = 86) (aproape jumatate din lista noastra de firme)

dupa o cautare pe google am aflat ca tarile din scandinavia sunt: suedia, norvegia si danemarca ,astfel cand vom cauta tari din regiunea scandinavica vom avea aceste 3 tari SI cand vom cauta tari nordice vom avea in plus si Finlanda si Islanda, astfel daca utilizatorul vrea sa aleaga una din cele 2 tari nordice (care nu sunt din scandinavia), acestea ii vor aparea in partea de jos

pentru tarile din eu o sa punem filtre pentru continente (europa, america, asia etc). o sa mai facem lista cu codurile tarilor de pe fiecare continent in parte, astfel cand utilizatorul cauta ex: Asia sa i apara toate tarile din asia din lista noastra de firme

legat de gb, ch vor intra in europa iar tr si ru vor intra atat in europa cat si in asia

In [33]:
df[["employee_count","revenue","year_founded"]].describe()

,employee_count,revenue,year_founded
count,2.890000e+02,3.840000e+02,346.000000
mean,2.509067e+04,1.146714e+10,1998.303468
std,1.379895e+05,1.282863e+11,33.156595
min,1.000000e+00,7.524000e+03,1856.000000
25%,5.000000e+00,2.483684e+06,1989.000000
50%,3.000000e+01,2.018448e+07,2011.000000
75%,2.603000e+03,1.592155e+09,2021.000000
max,2.100000e+06,2.500389e+12,2025.000000


In [34]:
(df["employee_count"] / 1000).round().sort_values(ascending=False)

214    2100.0
267     608.0
216     470.0
268     440.0
269     341.0
        ...  
437       NaN
438       NaN
439       NaN
440       NaN
442       NaN
Name: employee_count, Length: 477, dtype: float64

In [35]:
(df["revenue"] / 1000000000).round().sort_values(ascending=False)

214    2500.0
215     151.0
268     147.0
443     116.0
216      98.0
        ...  
432       NaN
437       NaN
439       NaN
440       NaN
442       NaN
Name: revenue, Length: 477, dtype: float64

In [36]:
df.is_public.value_counts()

is_public
False    436
True      41
Name: count, dtype: int64

In [37]:
df.loc[214]

website                                                   walmart.com
operational_name                                              Walmart
year_founded                                                   1962.0
address             {'country_code': 'us', 'latitude': 35.0739099,...
employee_count                                              2100000.0
revenue                                               2500388591944.0
primary_naics       {'code': '455211', 'label': 'Warehouse Clubs a...
description         Walmart Stores, Inc. is an American retail cor...
business_model      [Wholesale, E-commerce, Business-to-Business, ...
target_markets      [Consumer Goods, Automotive, Retail, Healthcar...
core_offerings      [Pet Supplies Retail, Omnichannel Retail Techn...
is_public                                                        True
secondary_naics                                                  None
country                                                            us
Name: 214, dtype: ob

dupa o scurta cautare pe google observam ca in momentul actual cea mai mare firma dupa venit este amazon si pe 2 walmart, walmart care are o cifra de 713 miliarde, noi avand 2500 miliarde. de aici deducem o posibila eroare de date

la employee_count observam o diferenta dintre medie si mediana (50%) este asa mare, deoarece sunt cateva firme in lista ex: walmart care trag media in sus, in timo ce mediana nu e afectata. concluzia firma tipica din date e mica si mediana este pe ce ne putem baza

la year_founded observam ca 75% dintre firme sunt infiintate in 2021 sau inainte, deci 25% sunt infiintate dupa 2021, adica foarte noi.

din cele 477 de firme avem doar 41 de firme publice avand 2 varinate, posibil o eroare de date (dar avand in vedere numarul mare de firme nepublice = 436, putem sa zicem ca nu ar fi asta) si varianta 2, avand in vedere ca vorbim de companii de peste tot din lume, observam o diferența majoră de legislație dintre jurisdicții și statutul juridic al companiei (privată vs. listată la bursă) sursa: google,
dar asta nu inseamna ca nu putem exclude si faptul ca majoritatea firmelor din lume sunt private, fapt care va afecta cererea 2

In [38]:
angajati=df.employee_count
angajati

0         NaN
1         NaN
2      7000.0
3        92.0
4        65.0
        ...  
472     329.0
473     310.0
474     310.0
475     290.0
476     276.0
Name: employee_count, Length: 477, dtype: float64

In [39]:
(angajati > 1000).sum()

np.int64(83)

In [40]:
venit=(df.revenue > 50000000).sum()
venit

np.int64(163)

In [41]:
an=(df.year_founded > 2018).sum()
an

np.int64(121)

In [42]:
(angajati < 200).sum()

np.int64(174)

In [43]:
df.employee_count.isna().sum()

np.int64(188)

In [44]:
((angajati >= 200) & (angajati <= 1000)).sum()

np.int64(32)

observam ca

firmele cu peste 1000 de angajati sunt 83

firmele cu venit de peste 50 mil sunt 163

firmele fondate dupa anul 2018 sunt 121

firmele cu mai putin de 200 de angajati sunt 174

mai avem si firmele cu valori NaN, acestea sunt vazute in df.employee_count.isna().sum() si se pot vedea ca nu sunt numarate nicaieri (83 + 174 = 257, 289 - 257 = 32 (((angajati >= 200) & (angajati <= 1000)).sum())) adica (83 + 174 + 32 + 188 (NaN) = 477)

concluzia este: cu un filtru pentru employee_count pot elimina acele 188 de firme fara sa vreau

In [45]:
df.primary_naics.apply(type).value_counts()

primary_naics
<class 'dict'>    477
Name: count, dtype: int64

In [46]:
df["industry"]=df.primary_naics.str.get("label")
df.industry

0                                   Petroleum Refineries
1      Other Chemical and Allied Products Merchant Wh...
2                                   Petroleum Refineries
3              Offices of Real Estate Agents and Brokers
4                   Nitrogenous Fertilizer Manufacturing
                             ...                        
472    Biological Product (except Diagnostic) Manufac...
473             Pharmaceutical Preparation Manufacturing
474             Pharmaceutical Preparation Manufacturing
475             Pharmaceutical Preparation Manufacturing
476             Pharmaceutical Preparation Manufacturing
Name: industry, Length: 477, dtype: object

In [47]:
df.industry.value_counts()

industry
Software Publishers                                       67
Turbine and Turbine Generator Set Units Manufacturing     44
Computer Systems Design Services                          30
Pharmaceutical Preparation Manufacturing                  25
Battery Manufacturing                                     24
                                                          ..
All Other Miscellaneous Manufacturing                      1
Cut and Sew Apparel Manufacturing (except Contractors)     1
Stationery Product Manufacturing                           1
Civic and Social Organizations                             1
In-Vitro Diagnostic Substance Manufacturing                1
Name: count, Length: 101, dtype: int64

In [48]:
df.industry.value_counts().head(20)

industry
Software Publishers                                                                                                   67
Turbine and Turbine Generator Set Units Manufacturing                                                                 44
Computer Systems Design Services                                                                                      30
Pharmaceutical Preparation Manufacturing                                                                              25
Battery Manufacturing                                                                                                 24
Engineering Services                                                                                                  22
Other Basic Inorganic Chemical Manufacturing                                                                          21
Financial Transactions Processing, Reserve, and Clearinghouse Activities                                              15
All Other Miscellaneous

In [49]:
df[df.industry.str.contains("Freight", case=False)].industry.value_counts()

Series([], Name: count, dtype: int64)

In [50]:
df[df.industry.str.contains("Transport", case=False)].industry.value_counts()

industry
Support Activities for Rail Transportation    1
Name: count, dtype: int64

In [51]:
df[df.industry.str.contains("Logistic", case=False)].industry.value_counts()

Series([], Name: count, dtype: int64)

In [52]:
df[df.industry.str.contains("Pharma", case=False)].industry.value_counts()

industry
Pharmaceutical Preparation Manufacturing    25
Name: count, dtype: int64

In [53]:
df[df.industry.str.contains("Software", case=False)].industry.value_counts()

industry
Software Publishers    67
Name: count, dtype: int64

In [54]:
df[df.industry.str.contains("Construction", case=False)].industry.value_counts()

industry
Commercial and Institutional Building Construction                  7
Other Heavy and Civil Engineering Construction                      5
Construction Machinery Manufacturing                                3
Highway, Street, and Bridge Construction                            2
Power and Communication Line and Related Structures Construction    2
Name: count, dtype: int64

In [55]:
df[df.industry.str.contains("Packag", case=False)].industry.value_counts()

Series([], Name: count, dtype: int64)

din cele 477 de firme cate avem in total, se regasesc doar 101 label-uri diferite, inseamna ca mai multe firme diferite se ocupa cu acelasi lucru

pentru cererea 1 ar intra logistic si cum se observa nu avem firme cand cautam STRICT dupa termenul logistic, posibil sa avem daca cautam dupa alti termeni conform NAICS sector 48-49 sursa: google

pentru cererea 4 avem „supply packaging for cosmetics” aici nu ne ajuta NAICS-ul, dar poate gasi producatori de ambalaje

In [56]:
df[df.industry.str.contains("Container", case=False)].industry.value_counts()

industry
Glass Container Manufacturing               3
Other Paperboard Container Manufacturing    1
Name: count, dtype: int64

In [57]:
df[df.industry.str.contains("Bottle", case=False)].industry.value_counts()

industry
Plastics Bottle Manufacturing    7
Bottled Water Manufacturing      1
Name: count, dtype: int64

In [58]:
df[df.industry.str.contains("Plastic", case=False)].industry.value_counts()

industry
All Other Plastics Product Manufacturing        8
Plastics Bottle Manufacturing                   7
Plastics Pipe and Pipe Fitting Manufacturing    1
Name: count, dtype: int64

In [59]:
df[df.industry.str.contains("Paper", case=False)].industry.value_counts()

industry
All Other Converted Paper Product Manufacturing    1
Other Paperboard Container Manufacturing           1
Name: count, dtype: int64

In [60]:
df[df.industry.str.contains("Glass", case=False)].industry.value_counts()

industry
Glass Container Manufacturing    3
Name: count, dtype: int64

dupa cautarea mai multor cuvinte cheie, am observat ca, furnizorii de ambalaje se gasesc 20 din 33, deoarece acele 13 (diferenta) fac alte chestii ex: Bottled Water Manufacturing sau sunt dubluri

In [61]:
df["naics"]=df.primary_naics.str.get("code")
df.naics

0      324110
1      424690
2      324110
3      531210
4      325311
        ...  
472    325414
473    325412
474    325412
475    325412
476    325412
Name: naics, Length: 477, dtype: object

In [62]:
df.naics.value_counts()

naics
513210    67
333611    44
541512    30
325412    25
335910    24
          ..
339999     1
315250     1
322230     1
813410     1
325413     1
Name: count, Length: 101, dtype: int64

In [63]:
df.naics.str.startswith(("48", "49")).sum()

np.int64(4)

In [64]:
df[df.naics.str.startswith(("48", "49"))].industry.value_counts()

industry
Support Activities for Rail Transportation    1
Port and Harbor Operations                    1
Postal Service                                1
General Warehousing and Storage               1
Name: count, dtype: int64

In [65]:
df[df.naics.str.startswith(("48", "49")) & (df.country == "ro")].industry.value_counts() 

industry
Support Activities for Rail Transportation    1
Port and Harbor Operations                    1
Postal Service                                1
General Warehousing and Storage               1
Name: count, dtype: int64

In [66]:
df[df.country == "ro"][["operational_name", "industry"]]

,operational_name,industry
0,Rompetrol,Petroleum Refineries
1,Unilever,Other Chemical and Allied Products Merchant Wh...
2,Rompetrol,Petroleum Refineries
3,CBRE Romania,Offices of Real Estate Agents and Brokers
4,TIMAC AGRO,Nitrogenous Fertilizer Manufacturing
5,OMV,Other Gasoline Stations
6,Compania Națională de Căi Ferate CFR,Support Activities for Rail Transportation
7,Mercedes-Benz Trucks & Buses Romania,New Car Dealers
8,Dacia,Automobile and Light Duty Motor Vehicle Manufa...
9,Fildas Trading,Drugs and Druggists' Sundries Merchant Wholesa...


transgaz, desi are trans in nume, compania se ocupa cu Natural Gas Distribution

se observa ca sunt firme ca nu se afla in sectorul de transport (Unilever, Fidas trading etc) si acestea fac transport din punct de vedere fizic dar nu presteaza servicii de transport

este o firma totusi care are ca si cuvinte in industry (Truck si Trailer), da, acestia fac dar firma este un furnizor pentru firmele de logistica, nu o firma de logistica

In [67]:
df.loc[21, "description"]

'STILL Material Handling Romania SRL is a Romanian company engaged in the development, production, and distribution of intralogistics equipment and systems. The company supplies and rents forklifts, transport and towing equipment, and internal logistics solutions, including static and dynamic racking, mobile racking, and automated guided vehicles. STILL also provides fleet management services, digital service management tools, and repair and maintenance solutions. The company serves a global market and operates production sites in Germany, Italy, China, France, and Brazil.'

concluzie

NAICS este un filtru bun ,dar nu este de ajuns, acesta poate pacali (ex Transgaz). Pentru cazurile de genul am nevoie de o descriere

In [68]:
for d in df.sample(10, random_state=1).description:
    print(d)
    print("---")

ENERCON is a Swedish company operating in the renewable energy sector, specializing in wind electric power generation and related services. The company is active in environmental services and the renewables & environment domain, with production facilities on three continents and market representation in more than 40 countries. ENERCON is committed to sustainability.
---
windainergy AB is a Swedish company engaged in renewable energy solutions, with a focus on small-scale wind turbines and AI-optimized wind power systems. The company serves local communities and small-scale renewable energy users worldwide, emphasizing sustainability and energy independence in its operations.
---
Skanska AB is a Swedish construction and development company engaged in project development, construction, and development services for residential, commercial, and infrastructure projects. The company operates primarily in the United States and serves a diverse range of clients across various sectors. Skanska 

observam ca toate incep cu numele companiei, dar apar informatii care nu sunt in alte campuri (ex Hunan Yueneng e exact cu cererea 12), windainergy nu are nici un camp care sa aprobe ca "AI-optimized wind power", ENERCON zice ca este o firma din Suedia dar (cautand o pe google aflam ca este una din Germania) deci si descrierile pot contine greseli fapt care face ca firma respectiva (si altele daca se regasesc in aceeasi situatie) sa fie reinvestigata sau sa fie lasata in partea de jos a listei, iar fimele Skanska si Decathlon desi u cuvantul logistics, ele nu sunt firme de logistica (la fel ca la STILL mai devreme). in plus, Skanska este o firma suedeza (sursa: google) dar zice ca "operates primarily in the United States" aici e simplu, pentru tara ma bazez pe country_code iar descrierea o folosesc ca semnal secundar

pentru cererile 9 si 10 nu apar nimic legat de crestere sau de tehnologii folosite ceea ce inseamna ca cererile respective sunt considerate grele, desi acestea pot si gasite prin alte cuvinte cheie (ex mai jos) 

legat de fast growing ne putem da seama doar daca ne uitam si in celelalte campuri (angajati, venit, anul infiintarii)

In [69]:
df[df.description.str.contains("Shopify", case=False)]

,website,operational_name,year_founded,address,employee_count,revenue,primary_naics,description,business_model,target_markets,core_offerings,is_public,secondary_naics,country,industry,naics


In [70]:
df[df.description.str.contains("e-commerce", case=False)]

,website,operational_name,year_founded,address,employee_count,revenue,primary_naics,description,business_model,target_markets,core_offerings,is_public,secondary_naics,country,industry,naics
20,brasov-industrialportfolio.ro,Brasov Industrial Portfolio,NaN,"{'country_code': 'ro', 'latitude': 44.450243, ...",NaN,1.000000e+09,"{'code': '493110', 'label': 'General Warehousi...",Brasov Industrial Portfolio is a Romanian comp...,"[Logistics/Transportation, Service Provider, B...","[Construction, Consumer Goods, E-commerce, Aut...","[High-Tech Logistics Facility Leasing, Consume...",False,None,ro,General Warehousing and Storage,493110
47,easykassa.at,EasyKassa,2016.0,"{'country_code': 'at', 'country_name': 'Austri...",1.0,3.177810e+05,"{'code': '522320', 'label': 'Financial Transac...",EasyKassa is an Austrian company that operates...,"[Service Provider, Business-to-Business]","[Retail, E-commerce, Payment Processing]","[Enterprise Software Solutions, Consumer Inter...",False,"{'code': '541512', 'label': 'Computer Systems ...",at,"Financial Transactions Processing, Reserve, an...",522320
58,arivpay.com,Ariv Pay,2020.0,"{'country_code': 'gb', 'country_name': 'United...",5.0,1.319900e+06,"{'code': '522320', 'label': 'Financial Transac...","ARIVPAY LIMITED, DBA Ariv Pay, is a British pr...","[Service Provider, Business-to-Business, Enter...","[Telecoms, Sports, Hospitality, Health, Corpor...","[Virtual Terminal Solutions, POS Payment Termi...",False,"{'code': '541512', 'label': 'Computer Systems ...",gb,"Financial Transactions Processing, Reserve, an...",522320
216,homedepot.com,The Home Depot,1978.0,"{'country_code': 'us', 'latitude': 33.8563351,...",470100.0,9.771205e+10,"{'code': '444110', 'label': 'Home Centers'}","The Home Depot, Inc. is an American multinatio...","[Retail, Service Provider, E-commerce, Busines...","[Industrial Supplies, Home Improvement]","[Fuel and Transportation Rental Services, Baby...",True,None,us,Home Centers,444110
217,usfoods.com,US Foods,1989.0,"{'country_code': 'us', 'latitude': 41.990926, ...",30000.0,5.227706e+10,"{'code': '424410', 'label': 'General Line Groc...","US Foods, Inc. is an American food distributio...","[Business-to-Business, E-commerce, Manufacturi...","[Hospitality, Food and Beverage, Education, He...","[Exclusive Brand Development Services, Food Sa...",True,None,us,General Line Grocery Merchant Wholesalers,424410
236,coopsleepgoods.com,Coop Sleep Goods,NaN,"{'country_code': 'us', 'latitude': 34.0581157,...",19000.0,1.064581e+10,"{'code': '337910', 'label': 'Mattress Manufact...","Coop Home Goods, LLC, known as Coop Sleep Good...","[Retail, E-commerce, Manufacturing, Business-t...","[Home Goods, Healthcare]","[Adjustable Pillow Manufacturing, Mattress Top...",False,None,us,Mattress Manufacturing,337910
240,harborfreight.com,Harbor Freight,1977.0,"{'country_code': 'us', 'latitude': 34.14171373...",28000.0,9.357146e+09,"{'code': '444140', 'label': 'Hardware Retailers'}",Harbor Freight is an American retailer engaged...,"[Retail, E-commerce, Business-to-Consumer]","[Home Improvement, Automotive]","[Community Education Support Services, Tool an...",False,None,us,Hardware Retailers,444140
265,flextribe.co,Flextribe,NaN,"{'country_code': 'us', 'latitude': 30.34148, '...",NaN,NaN,"{'code': '322219', 'label': 'Other Paperboard ...",Flextribe is a company engaged in eco-friendly...,"[Manufacturing, E-commerce, Business-to-Busine...","[Photography, Electronics, Cosmetics, Home Décor]","[Eco-Friendly Packaging Solutions, Direct-to-C...",False,None,us,Other Paperboard Container Manufacturing,322219
275,telefonica.com,Telefónica Venture Builder,NaN,"{'country_code': 'es', 'latitude': 40.5130306,...",100870.0,1.687307e+10,"{'code': '541512', 'label': 'Computer Systems ...","Telefónica Innovación Digital, S.L.U., known a...","[Service Provider, Business-to-Business, Enter...","[Telecommunications, Media and Entertainment, ...","[Telecommunications Services, Digital Transfor...",True,None,es,Computer Sys

In [71]:
df[df.description.str.contains("online store", case=False)]

,website,operational_name,year_founded,address,employee_count,revenue,primary_naics,description,business_model,target_markets,core_offerings,is_public,secondary_naics,country,industry,naics
233,drinkarizona.com,DrinkAriZona,1992.0,"{'country_code': 'us', 'latitude': 39.102322, ...",10000.0,1.100833e+10,"{'code': '312111', 'label': 'Soft Drink Manufa...","AriZona Beverages USA, LLC., known as DrinkAri...","[Subscription-Based, Manufacturing, E-commerce...",[Food and Beverage],"[Alcoholic Beverage Manufacturing, Beverage Su...",False,None,us,Soft Drink Manufacturing,312111
234,decathlon.it,Decathlon,1993.0,"{'country_code': 'it', 'latitude': 45.6054541,...",105000.0,1.099349e+10,"{'code': '459110', 'label': 'Sporting Goods Re...",DECATHLON ITALIA S.r.l. Unipersonale is an Ita...,"[Wholesale, Manufacturing, E-commerce, Busines...",[Sports and Recreation],[Quality and Environmental Management Consulti...,False,None,it,Sporting Goods Retailers,459110
237,pez.com,PEZ Candy,1927.0,"{'country_code': 'us', 'latitude': 41.26377215...",110.0,1.057338e+10,"{'code': '311340', 'label': 'Nonchocolate Conf...","PEZ Candy, Inc., DBA PEZ Candy, is an American...","[Wholesale, Manufacturing, E-commerce, Busines...",[Food and Beverage],"[Merchandise Manufacturing, PEZ Dispenser Dist...",False,None,us,Nonchocolate Confectionery Manufacturing,311340
243,aoteanz.com,Aotea Wellness,NaN,"{'country_code': 'nz', 'latitude': -36.8430427...",47.0,8.530423e+09,"{'code': '325620', 'label': 'Toilet Preparatio...","Aotea New Zealand Online Limited, known as Aot...","[Wholesale, Manufacturing, E-commerce, Busines...","[Tourism, Wellness and Personal Care]","[Community Engagement Services, Destination Gi...",False,None,nz,Toilet Preparation Manufacturing,325620


In [72]:
df[df.business_model.astype(str).str.contains("E-commerce", case=False)]

,website,operational_name,year_founded,address,employee_count,revenue,primary_naics,description,business_model,target_markets,core_offerings,is_public,secondary_naics,country,industry,naics
214,walmart.com,Walmart,1962.0,"{'country_code': 'us', 'latitude': 35.0739099,...",2100000.0,2.500389e+12,"{'code': '455211', 'label': 'Warehouse Clubs a...","Walmart Stores, Inc. is an American retail cor...","[Wholesale, E-commerce, Business-to-Business, ...","[Consumer Goods, Automotive, Retail, Healthcar...","[Pet Supplies Retail, Omnichannel Retail Techn...",True,None,us,Warehouse Clubs and Supercenters,455211
215,dell.com,Dell Technologies,1984.0,"{'country_code': 'au', 'latitude': -33.7948928...",120000.0,1.514513e+11,"{'code': '334111', 'label': 'Electronic Comput...","Dell Inc., known as Dell Technologies, is an A...","[Wholesale, Manufacturing, E-commerce, Busines...","[Telecommunications, Energy, Retail, Smart Cit...","[Technology Product Retail and Distribution, S...",False,None,au,Electronic Computer Manufacturing,334111
216,homedepot.com,The Home Depot,1978.0,"{'country_code': 'us', 'latitude': 33.8563351,...",470100.0,9.771205e+10,"{'code': '444110', 'label': 'Home Centers'}","The Home Depot, Inc. is an American multinatio...","[Retail, Service Provider, E-commerce, Busines...","[Industrial Supplies, Home Improvement]","[Fuel and Transportation Rental Services, Baby...",True,None,us,Home Centers,444110
217,usfoods.com,US Foods,1989.0,"{'country_code': 'us', 'latitude': 41.990926, ...",30000.0,5.227706e+10,"{'code': '424410', 'label': 'General Line Groc...","US Foods, Inc. is an American food distributio...","[Business-to-Business, E-commerce, Manufacturi...","[Hospitality, Food and Beverage, Education, He...","[Exclusive Brand Development Services, Food Sa...",True,None,us,General Line Grocery Merchant Wholesalers,424410
218,decathlon.com,Decathlon,1976.0,"{'country_code': 'fr', 'latitude': 50.6456712,...",101000.0,4.325797e+10,"{'code': '459110', 'label': 'Sporting Goods Re...","Decathlon SE, DBA Decathlon, is a French compa...","[Retail, E-commerce, Manufacturing, Business-t...","[Apparel and Footwear, Sports and Recreation]","[Clothing and Sportswear Retail, Sporting Good...",False,None,fr,Sporting Goods Retailers,459110
219,forever21.com,Forever 21,1984.0,"{'country_code': 'us', 'latitude': 40.7517868,...",43000.0,3.640042e+10,"{'code': '455110', 'label': 'Department Stores'}","F21 IPCO, LLC, known as Forever 21, is an Amer...","[Retail, E-commerce, Business-to-Consumer]","[Beauty and Personal Care, Home Goods, Apparel...","[Beauty Products Retail, Fashion Product Marke...",False,None,us,Department Stores,455110
220,nothingnowhere.net,"nothing,nowhere",2005.0,"{'country_code': 'us', 'latitude': 38.850125, ...",29000.0,3.101114e+10,"{'code': '512250', 'label': 'Record Production...","nothing,nowhere is an American music company s...","[E-commerce, Retail, Entertainment, Service Pr...",[Music Industry],"[Music Content Distribution, Live Performance ...",False,None,us,Record Production and Distribution,512250
221,nipponpaint.com.sg,Nippon Paint Singapore,1962.0,"{'country_code': 'sg', 'latitude': 1.3278391, ...",6.0,3.024763e+10,"{'code': '325510', 'label': 'Paint and Coating...",Nippon Paint Singapore Co. Pte Ltd is a Singap...,"[Wholesale, Manufacturing, E-commerce, Busines...","[Construction, Consumer Goods, Automotive]","[Environmentally Friendly Paint Development, C...",False,None,sg,Paint and Coating Manufacturing,325510
222,gillette.com,Gillette,1901.0,"{'country_code': 'us', 'latitude': 39.10303995...",29000.0,2.860296e+10,"{'code': '339999', 'label': 'All Other Miscell...",Gillette Company LLC is an American company en...,"[Subscription-Based, Manufacturing, E-commerce...",[Personal Care],"[Intimate Shaving Product Manufacturing, Perso...",False,None,us,All Other Miscellaneous Manufacturing,339999
223,coles.com.au,Coles,1914.0,"{'country_code': 'au', 'latitude': -37.8453794...",120000.0,2.726001e+10,"{'code': '44511

In [73]:
len(df[df.business_model.astype(str).str.contains("E-commerce", case=False)])

38

au iesit in total 38 de firme cu eticheta E-commerce desi unele precum (walmart, dell, gillette etc) nu sunt facute cu shopify ele doar sunt firme care vand si online, eticheta fiind una prea larga, deci ea nu e suficienta pentru cererea 10

In [74]:
df[df.country == "ro"][["operational_name", "industry"]]

,operational_name,industry
0,Rompetrol,Petroleum Refineries
1,Unilever,Other Chemical and Allied Products Merchant Wh...
2,Rompetrol,Petroleum Refineries
3,CBRE Romania,Offices of Real Estate Agents and Brokers
4,TIMAC AGRO,Nitrogenous Fertilizer Manufacturing
5,OMV,Other Gasoline Stations
6,Compania Națională de Căi Ferate CFR,Support Activities for Rail Transportation
7,Mercedes-Benz Trucks & Buses Romania,New Car Dealers
8,Dacia,Automobile and Light Duty Motor Vehicle Manufa...
9,Fildas Trading,Drugs and Druggists' Sundries Merchant Wholesa...


In [75]:
df.loc[1, "description"]

'Unilever South Central Europe SA, DBA Unilever, is a Romanian company specialized in the distribution and retail of consumer goods, with a focus on personal care and household products. The company serves the Romanian market and is part of the Unilever Group, which operates globally. Unilever România also engages in digital initiatives, including platforms for suppliers and customers, and is committed to sustainability and social responsibility.'

In [76]:
df.loc[0, "description"]

'Rompetrol is a Romanian company specialized in petroleum refining, petrochemical operations, and the distribution of fuel products. The company operates as a subsidiary of KMG International and manages integrated refineries in Romania, Moldova, Bulgaria, and Georgia, serving the automotive, industrial, and energy sectors. Rompetrol also provides industrial products, wholesale fuel supply, and e-Mobility services, and is certified for quality, health, safety, and environmental management.'

In [77]:
df.loc[2, "description"]

'Rompetrol is a Romanian energy company engaged in refining, petrochemicals, and retail operations, as well as providing well services in the upstream sector. The company supplies fuels and related products to both individual and business clients in Romania, Moldova, Bulgaria, Georgia, and Kazakhstan, and is involved in the development of energy projects through its participation in the Kazakh-Romanian Energy Investment Fund. Rompetrol also manages a supply chain and logistics network to support its operations and is committed to sustainability and decarbonization initiatives.'

In [78]:
df[df.country == "ro"][["operational_name", "industry", "description"]]

,operational_name,industry,description
0,Rompetrol,Petroleum Refineries,Rompetrol is a Romanian company specialized in...
1,Unilever,Other Chemical and Allied Products Merchant Wh...,"Unilever South Central Europe SA, DBA Unilever..."
2,Rompetrol,Petroleum Refineries,Rompetrol is a Romanian energy company engaged...
3,CBRE Romania,Offices of Real Estate Agents and Brokers,CBRE Romania is a Romanian company engaged in ...
4,TIMAC AGRO,Nitrogenous Fertilizer Manufacturing,TIMAC AGRO S.R.L. is a French agricultural com...
5,OMV,Other Gasoline Stations,OMV PETROM Marketing SRL is a Romanian company...
6,Compania Națională de Căi Ferate CFR,Support Activities for Rail Transportation,"Compania Națională de Căi Ferate ”CFR” – SA, k..."
7,Mercedes-Benz Trucks & Buses Romania,New Car Dealers,"Daimler Truck & Bus Romania S.R.L., DBA Merced..."
8,Dacia,Automobile and Light Duty Motor Vehicle Manufa...,"Renault Commercial Roumanie S.R.L., DBA Dacia,..."
9,Fildas Trading,Drugs and Druggists' Sundries Merchant Wholesa...,Fildas Trading SRL is a Romanian company engag...


In [79]:
df.loc[22, "description"]

'AQUA INVEST, known as AQUA CENTER, is a Romanian company engaged in the production and export of composite pipe products, including composite pipes, PVC pipes, and PVC fittings, serving the construction, water supply, and drainage sectors. The company operates a logistics and production center and exports its products to over 40 countries, with a significant market presence in China. AQUA CENTER is ISO certified and is involved in the development of Yongtong Industrial Park and the expansion of production capacity.'

In [80]:
df.loc[23, "description"]

'S.C. European Drinks S.A., DBA European Drinks, is a Romanian company specialized in the production and distribution of mineral water, carbonated water, energy drinks, beer, and various soft drinks. The company operates as a holding with subsidiaries in the food and drink sector, serving both domestic and international markets. European Drinks also engages in community support initiatives and participates in international industry events.'

In [81]:
df.loc[65, "description"]

'HR Sincron SRL, DBA Sincron HR, is a Romanian software company specialized in providing HR software solutions, including recruitment, talent management, and HR analytics. The company offers a comprehensive platform that covers areas such as recruitment, onboarding, staffing, time and attendance, payroll, performance management, training management, employee self-service, internal communication, and HR analytics. Sincron HR serves clients across Europe and North America, supporting businesses of various sizes in streamlining HR processes and operational efficiency.'

In [82]:
df.loc[329, "description"]

'EcoExpert Green Consulting S.R.L is a Romanian consulting company engaged in environmental, social, and governance (ESG) solutions. The company provides services such as carbon footprint calculation, sustainability reporting, and Net Zero compliance, utilizing technologies including Blockchain, AI, and sensors. EcoExpert Green Consulting serves a diverse client base, including large enterprises, start-ups, and financial institutions, and offers modular ESG solutions to support companies in achieving sustainability objectives. The company also operates a platform for ESG reporting and maintains a network of partners and clients to facilitate access to sustainability tools and resources.'

se poate observa intai de toate ca rompetrol se repeta ceea ce ma face sa mi pun un semn de intrebare


| Index | Firma | Industrie (NAICS) | Eticheta | De ce |
|---|---|---|---|---|
| 0 | Rompetrol | Petroleum Refineries | NU | rafinare si distributie de combustibil propriu |
| 1 | Unilever | Chemical Products Merchant Wholesalers | NU | angrosist, transporta doar propriile produse |
| 2 | Rompetrol (dublura) | Petroleum Refineries | NU | are "logistics" in descriere, dar "to support its operations" -> logistica interna |
| 3 | CBRE Romania | Real Estate Agents and Brokers | NU | servicii imobiliare |
| 4 | TIMAC AGRO | Nitrogenous Fertilizer Manufacturing | NU | produce ingrasaminte |
| 5 | OMV | Gasoline Stations | NU | distribuie propriul combustibil |
| 6 | CFR | Support Activities for Rail Transportation | DA | infrastructura si servicii de transport feroviar |
| 7 | Mercedes-Benz Trucks & Buses | New Car Dealers | NU | vinde si repara camioane -> furnizor pentru logistica |
| 8 | Dacia | Automobile Manufacturing | NU | produce masini; descrierea nu spune nimic de transport |
| 9 | Fildas Trading | Drugs Merchant Wholesalers | NU | angrosist |
| 10 | Portul Constanta | Port and Harbor Operations | DA | operator portuar |
| 11 | Bunge Romania | Oilseed Processing | NU | angrosist de marfuri agricole, transporta doar propriile produse |
| 12 | Romgaz | Natural Gas Extraction | DA | are "storage" si "transportation services" (caz la limita) |
| 13 | METRO Romania | Grocery Merchant Wholesalers | NU | angrosist |
| 14 | Transgaz | Natural Gas Distribution | NU | numele sugereaza transport, dar distribuie gaze |
| 15 | Posta Romana | Postal Service | DA | servicii postale si de curierat |
| 16 | OSCAR | Petroleum Products Merchant Wholesalers | NU | distribuie propriul combustibil |
| 17 | Prutul | Fats and Oils Refining | NU | produce uleiuri |
| 18 | Valtec | Petroleum Products Merchant Wholesalers | NU | angrosist de combustibil |
| 19 | Valbur | Industrial Supplies Merchant Wholesalers | DA | "physical movement" de marfuri pentru clienti (caz la limita) |
| 20 | Brasov Industrial Portfolio | General Warehousing and Storage | DA | depozitare si logistica last-mile |
| 21 | STILL | Industrial Truck Manufacturing | NU | produce stivuitoare -> furnizor pentru logistica |
| 22 | AQUA CENTER | Plastics Pipe Manufacturing | NU | produce tevi |
| 23 | European Drinks | Bottled Water Manufacturing | NU | produce bauturi |
| 65 | Sincron HR | Software Publishers | NU | software HR |
| 329 | EcoExpert Green Consulting | Environmental Consulting | NU | consultanta de mediu |

total: 6 DA - 0 POATE - 20 NU

regula folosita: firma care face logistica pentru altii = DA; firma care isi transporta doar propriile produse sau vinde echipamente pentru logistica = NU.

cazuri la limita (Romgaz si Valbur) au elemente de transport sau depozitare, dar nu e clar daca e afacerea lor principala.

Etapa 2 – Desfacerea cererilor

Cererea 5: "Construction companies in the United States with revenue over $50 million"

| Constrangere | Tip | Camp(uri) | Probleme / decizii |
|---|---|---|---|
| "Construction companies" | INTERPRETARE | `industry`, `naics`, `description` | firme de constructii vs firme care vand materiale sau echipamente pentru constructii (capcana "furnizor pentru") |
| "in the United States" | FILTRU | `country == "us"` | ma bazez pe sediu (`country_code`), descrierea e semnal secundar |
| "revenue over $50 million" | FILTRU | `revenue > 50_000_000` | 19,5% lipsa -> firmele fara venit nu le elimin, le cobor in clasament |

Cererea 2: "Public software companies with more than 1,000 employees"

| Constrangere | Tip | Camp(uri) | Probleme / decizii |
|---|---|---|---|
| "Software companies" | INTERPRETARE | `industry`, `naics`, `description` | firme de software, este iar o capcana, deorece pot exista firme de harware cand vand software (ex Dell) |
| "public" | FILTRU | `is_public == True` | o afirmatie buna filtrul are 0% lipsa deci e de incredere, ajuta la selectare |
| "with more than 1,000 employees" | FILTRU | `employee_count > 1000` | 39.4%, daca elimin raman fara 4 din 10 firme aproape, le coboram in clasament |

Cererea 3: "Food and beverage manufacturers in France"

| Constrangere | Tip | Camp(uri) | Probleme / decizii |
|---|---|---|---|
| "Food and beverage manufacturers" | FILTRU | `naics` incepe cu 311 (food) sau 312 (beverage) | filtrul pe cod exclude automat furnizorii (ambalaje) si angrosistii (METRO), spre deosebire de cautarea dupa cuvinte. Problema: 312 include si tutunul, care trebuie exclus. Produit en Bretagne are "agro-food" in descriere, dar e o asociatie, nu un producator -> exclusa corect de filtru (similaritate diferita de relevanta) |
| "in France" | FILTRU | `country == "fr"` | la fel ma bazez intai pe `country_code`, descrierea e secundara |

Cererea 6: "Pharmaceutical companies in Switzerland"

| Constrangere | Tip | Camp(uri) | Probleme / decizii |
|---|---|---|---|
| "Pharmaceutical companies" | FILTRU (NAICS 3254) + INTERPRETARE | `naics` incepe cu 3254, `description` | 3254 prinde 32 de firme elvetiene. "Companies" e mai larg decat "manufacturers", deci verific si firmele din afara lui 3254. Decizii la limita: PSI CRO -> NU (face studii clinice pentru firme farma, nu produce medicamente); Drugs for Neglected Diseases initiative -> DA (dezvolta medicamente, chiar daca e non-profit si nu e in 3254); Lonza -> DA (produce medicamente, chiar daca pentru alte firme); Nestle Health Science -> NU (nutritie si suplimente, desi e in 3254); AliveDx -> NU (teste de diagnostic, nu medicamente) |
| "in Switzerland" | FILTRU | `country == "ch"` | ma bazez pe `country_code`, descrierea e secundara |

Cererea 1: "Logistic companies in Romania"

| Constrangere | Tip | Camp(uri) | Probleme / decizii |
|---|---|---|---|
| "Logistic companies" | FILTRU (NAICS 48-49) + INTERPRETARE | `naics`, `description` | 48-49 prinde CFR, Portul Constanta, Posta Romana, Brasov Industrial Portfolio, dar rateaza Valbur si Romgaz. Capcane: firme cu "logistics"/"distribution" in descriere pentru propriile operatiuni (Rompetrol, OMV), angrosisti (Unilever, METRO, Bunge), furnizori pentru logistica (STILL, Mercedes Trucks), nume care pacalesc (Transgaz). Regula: logistica pentru altii = DA |
| "in Romania" | FILTRU | `country == "ro"` | ma bazez pe `country_code`, descrierea e secundara |

Set de evaluare: etichetat manual, 6 DA din 26 firme romanesti.

Cererea 4: "Companies that could supply packaging materials for a direct-to-consumer cosmetics brand"

| Constrangere | Tip | Camp(uri) | Probleme / decizii |
|---|---|---|---|
| "supply packaging materials" | FILTRU (NAICS) + INTERPRETARE | `naics` (ambalaje: sticla, plastic, carton), `description` | intra si producatorii, si distribuitorii de ambalaje, pentru ca cererea zice "supply", nu "manufacture". Unele coduri sunt prea largi ("All Other Plastics Product") -> trebuie citita descrierea |
| "for a ... cosmetics brand" | INTERPRETARE | `description`, `target_markets` | decizie: firma trebuie sa mentioneze cosmetice (sau beauty / personal care) in descriere sau in target_markets |
| capcana | - | - | firmele de cosmetice propriu-zise NU intra: ele sunt clientul, nu furnizorul (exact eroarea embeddings din cerinta) |
| "direct-to-consumer" | INTERPRETARE (slaba) | `description` | e o caracteristica a clientului, nu a furnizorului -> greu de verificat; o folosesc doar ca bonus (ex: furnizori care accepta comenzi mici) |


Cererea 7: "B2B SaaS companies providing HR solutions in Europe"

| Constrangere | Tip | Camp(uri) | Probleme / decizii |
|---|---|---|---|
| "HR solutions" | FILTRU (NAICS) + INTERPRETARE | `naics`, `description`, `core_offerings` | intelegere larga: software HR, consultanta HR, recrutare, salarizare (payroll) |
| "SaaS" | FILTRU + INTERPRETARE | `business_model`, `description` | verific in ambele campuri. Consultanta HR pura nu e SaaS -> intra doar daca ofera si software |
| "B2B" | FILTRU | `business_model` contine "B2B" | software HR se vinde aproape mereu firmelor, deci e un filtru usor |
| "in Europe" | FILTRU | `country` in lista de coduri europene | decizia de ieri: gb si ch intra; tr si ru intra si in Europa, si in Asia |

Cererea 8: "Clean energy startups founded after 2018 with fewer than 200 employees"

| Constrangere | Tip | Camp(uri) | Probleme / decizii |
|---|---|---|---|
| "Clean energy" | FILTRU (NAICS) + INTERPRETARE | `naics`, `description`, `core_offerings` | intra: solar/eolian, baterii si stocare, vehicule electrice. Nu intra: consultanta pentru eficienta energetica |
| "startups" | INTERPRETARE | `description` | pe langa an si numar de angajati, firma trebuie sa dezvolte un produs sau o tehnologie noua -> caut in descriere cuvinte cheie ("developing", "innovative", "new technology"). Risc: cuvintele cheie pot da rezultate gresite (similaritate diferita de relevanta) |
| "founded after 2018" | FILTRU | `year_founded > 2018` | 27,5% lipsa -> nu elimin, cobor in clasament |
| "fewer than 200 employees" | FILTRU | `employee_count < 200` | 39,4% lipsa -> nu elimin, cobor in clasament |
| combinatie | - | - | cea mai riscanta cerere din punct de vedere al datelor lipsa: depinde de 2 campuri cu lipsuri. O firma poate sa nu aiba anul, numarul de angajati sau pe amandoua |

Cererea 9: "Fast-growing fintech companies competing with traditional banks in Europe"

| Constrangere | Tip | Camp(uri) | Probleme / decizii |
|---|---|---|---|
| "fintech companies" | FILTRU (NAICS) + INTERPRETARE | `naics` (ex: 5223 - procesare plati), `description` | firme care ofera servicii financiare prin tehnologie |
| "competing with traditional banks" | INTERPRETARE | `description`, `core_offerings` | intra firmele care ofera direct clientilor servicii pe care le dau de obicei bancile (plati, conturi, credite, carduri). NU intra firmele care vand software bancilor (ex: SAP) -> sunt furnizori pentru banci, nu concurenti (capcana "furnizor pentru") |
| "fast-growing" | INTERPRETARE (slaba) | `description` | nu exista un camp pentru crestere -> caut cuvinte cheie ("growth", "fast-growing", "expanding", "rapidly"). Risc: multe firme se descriu singure ca fiind "in crestere", deci semnalul e slab -> il folosesc doar pentru clasament, nu ca filtru strict |
| "in Europe" | FILTRU | `country` in lista de coduri europene | aceeasi lista ca la cererea 7 |

Cererea 10: "E-commerce companies using Shopify or similar platforms"

| Constrangere | Tip | Camp(uri) | Probleme / decizii |
|---|---|---|---|
| "E-commerce companies" | FILTRU + INTERPRETARE | `business_model` contine "E-commerce", `description` | firma trebuie sa vanda online ca afacere principala, nu doar ca un canal in plus. Eticheta "E-commerce" din business_model e prea larga (38 de firme, include Walmart, Dell, Gillette) -> trebuie confirmata din descriere |
| "using Shopify or similar platforms" | INTERPRETARE (aproximare) | `description`, `employee_count` | Shopify nu apare nicaieri in date -> constrangerea nu poate fi verificata direct. Aproximare: branduri mici care vand direct online (D2C), pentru ca de obicei ele folosesc platforme gata facute ca Shopify. Limita explicata in writeup |
| capcana | - | - | firmele mari (Walmart, Dell) NU intra: au magazine online proprii, construite intern, nu pe Shopify. Firmele care servesc e-commerce (ex: Brasov Industrial Portfolio, depozitare) NU intra: sunt furnizori pentru e-commerce |

Cererea 11: "Renewable energy equipment manufacturers in Scandinavia"

| Constrangere | Tip | Camp(uri) | Probleme / decizii |
|---|---|---|---|
| "Renewable energy equipment manufacturers" | FILTRU (NAICS) + INTERPRETARE | `naics` (ex: 333611 - Turbine Manufacturing), `description` | intra doar firmele care produc echipamente complete (turbine eoliene, panouri solare, invertoare). NU intra: firmele care construiesc sau opereaza parcuri eoliene/solare (nu produc echipamente) si producatorii de componente (pale, cutii de viteze) |
| capcana | - | - | "Turbine Manufacturing" poate include si turbine pentru gaze sau abur, care nu sunt regenerabile -> trebuie verificata descrierea |
| "in Scandinavia" | FILTRU | `country` in [se, no, dk] + [fi, is] mai jos in clasament | decizia de ieri: Scandinavia stricta = Suedia, Norvegia, Danemarca; Finlanda si Islanda intra, dar mai jos in clasament |
| problema de date | - | - | ENERCON e descrisa ca "Swedish", dar e germana -> ma bazez pe `country_code` |

Cererea 12: "Companies that manufacture or supply critical components for electric vehicle battery production"

| Constrangere | Tip | Camp(uri) | Probleme / decizii |
|---|---|---|---|
| "critical components for EV battery production" | FILTRU (NAICS) + INTERPRETARE | `naics` (ex: 335910 - Battery Manufacturing), `description`, `core_offerings` | intra: producatorii de baterii, producatorii de componente (catozi, anozi, separatori, electroliti) si firmele care extrag materii prime critice (litiu, nichel, cobalt). NU intra: firmele care fac echipamente pentru fabricile de baterii (utilaje) |
| "manufacture or supply" | INTERPRETARE | `description` | intra si producatorii, si furnizorii/distribuitorii (ca la cererea 4) |
| "electric vehicle" | INTERPRETARE | `description`, `target_markets` | bateriile pot fi si pentru alte domenii (telefoane, stocare energie). Ideal, descrierea sau target_markets mentioneaza vehicule electrice / automotive |
| rol in ecosistem | - | - | cererea e despre lantul de aprovizionare: conteaza cui ii vinde firma (producatorilor de baterii pentru EV), nu doar ce produce. Exemplu bun: Hunan Yueneng ("supplies battery manufacturers") |
| fara tara | - | - | cererea nu are filtru geografic |

In [83]:
df[df.naics.str.startswith(("311", "312")) & (df.country == "fr")].industry.value_counts() 

industry
Soft Drink Manufacturing                      9
Wineries                                      4
Fruit and Vegetable Canning                   3
All Other Miscellaneous Food Manufacturing    2
Specialty Canning                             1
Meat Processed from Carcasses                 1
Name: count, dtype: int64

In [84]:
df[df.country == "fr"][["operational_name", "industry"]]

,operational_name,industry
62,European Pay,"Financial Transactions Processing, Reserve, an..."
69,Kapsoon,Software Publishers
71,Asys,Software Publishers
73,quarksUp,Software Publishers
75,NOVRH,Software Publishers
83,Virtualia,Software Publishers
84,Kammi,Software Publishers
86,Cantoriel,Software Publishers
94,Sun RH,Software Publishers
97,Staff & Go,Software Publishers


In [85]:
df[df.country == "ch"][["operational_name", "industry"]]

,operational_name,industry
74,Novative,Human Resources Consulting Services
81,HR Campus,Human Resources Consulting Services
88,Foederis,Software Publishers
127,Deepstributed,Software Publishers
297,Solarschmiede,Engineering Services
301,Terzic Urban Engineering,Engineering Services
304,visusta,Environmental Consulting Services
312,MA-Engineering,Engineering Services
374,Greenbatt Technology,Battery Manufacturing
443,Novartis,Pharmaceutical Preparation Manufacturing


In [86]:
df[df.naics.str.startswith(("3254")) & (df.country == "ch")].industry.value_counts() 


industry
Pharmaceutical Preparation Manufacturing                25
Biological Product (except Diagnostic) Manufacturing     4
Medicinal and Botanical Manufacturing                    2
In-Vitro Diagnostic Substance Manufacturing              1
Name: count, dtype: int64

Set de evaluare - Cererea 6: "Pharmaceutical companies in Switzerland"

Regula: intra cine dezvolta sau produce medicamente, indiferent de NAICS sau de forma de organizare (inclusiv pentru alte firme).

| Index | Firma | Eticheta | De ce |
|---|---|---|---|
| 443 | Novartis | DA | dezvolta si produce medicamente |
| 444 | Lonza | DA | produce medicamente pentru alte firme (CDMO) |
| 445 | Octapharma | DA | terapii din proteine umane |
| 447 | Ferring | DA | dezvolta si produce medicamente |
| 448 | Galderma | DA | tratamente dermatologice |
| 449 | Sumitomo Pharma Switzerland | DA | produce medicamente |
| 450 | Lonza Capsules & Health Ingredients | DA | incapsulare pentru firme farma (partea de nutritie nu conteaza) |
| 451 | Siegfried | DA | ingrediente active pentru medicamente (CDMO) |
| 453 | Acino | DA | dezvolta si produce medicamente |
| 454 | CordenPharma | DA | ingrediente active pentru medicamente (CDMO) |
| 455 | PolyPeptide | DA | ingrediente active pe baza de peptide (CDMO) |
| 456 | Idorsia | DA | dezvolta medicamente |
| 457 | Stallergenes Greer | DA | tratamente pentru alergii |
| 458 | IBSA | DA | produce medicamente |
| 459 | Bachem | DA | peptide pentru medicamente |
| 460 | Alcon | DA | medicamente pentru ochi (si echipamente chirurgicale) |
| 461 | CARBOGEN AMCIS | DA | ingrediente active pentru medicamente (CDMO) |
| 463 | OM Pharma | DA | dezvolta medicamente |
| 464 | MSD | DA | medicamente si vaccinuri |
| 466 | Drugs for Neglected Diseases initiative | DA | dezvolta medicamente (non-profit, in afara NAICS 3254) |
| 467 | Debiopharm | DA | dezvolta si produce medicamente |
| 468 | CRISPR Therapeutics | DA | terapii genice |
| 470 | Helsinn | DA | produce medicamente |
| 472 | Celonic | DA | produce medicamente biologice (CDMO) |
| 473 | ADC Therapeutics | DA | tratamente pentru cancer |
| 474 | Tillotts Pharma | DA | medicamente gastroenterologice |
| 475 | KD Pharma Group | DA | CDMO pe partea farmaceutica (omega-3 nu conteaza) |
| 476 | A. Menarini | DA | produce medicamente |
| 446 | Nestle Health Science | NU | nutritie, desi e in NAICS 3254 |
| 452 | PSI CRO | NU | testeaza medicamente (studii clinice), nu le produce |
| 462 | Teoxane | NU | filler-e estetice, "medical equipment" |
| 465 | AbelaPharm | NU | suplimente alimentare |
| 469 | AliveDx | NU | teste de diagnostic, nu medicamente |
| 471 | TRB Chemedica | NU | produse cu acid hialuronic, la limita cu dispozitivele medicale |
| 74, 81, 88, 127, 297, 301, 304, 312, 374 | Novative, HR Campus, Foederis, Deepstributed, Solarschmiede, Terzic, visusta, MA-Engineering, Greenbatt | NU | fara legatura cu farma |

Total: 28 DA - 15 NU (din 43 firme elvetiene)

Observatii:
- NAICS 3254 prinde 32 de firme: 27 sunt DA, 5 sunt NU (Nestle, Teoxane, AbelaPharm, AliveDx, TRB Chemedica)
- o singura firma DA e in afara lui 3254: DNDi
- concluzie: doar filtrul NAICS 3254 ar nimeri 27 din 28 de firme bune, dar ar prinde si 5 gresite
- cazuri la limita: Alcon, Galderma (medicamente + alte produse), Lonza Capsules, KD Pharma (farma + nutritie), TRB Chemedica

Set de evaluare - Cererea 12: "Companies that manufacture or supply critical components for electric vehicle battery production"

Reguli:
- DA: producatori de baterii, de componente (catozi, anozi, electroliti, separatori) si de materii prime (litiu)
- NU: echipamente pentru fabricile de baterii; firme care doar instaleaza sisteme solare/stocare
- LA LIMITA: producatori de baterii care nu mentioneaza EV (doar stocare); firme cu activitati mixte
- materialele (catozi, anozi etc.) raman DA chiar fara mentiune de EV, pentru ca sunt aceleasi pentru masini si pentru stocare

Candidati: 69 de firme cu legatura cu bateriile (din descriere sau NAICS)

| Grupa | Firme | Eticheta | De ce |
|---|---|---|---|
| Materiale si componente | E-Lyte, Blue New Energy Technology, SolidEdge Solution, Sili Energy, Hisun, Rongbai, RUSTOR, FeovaEnergy, Li Hongte, Altmin, Dongdao, Yide, POLYENTECH, BTR New Material, Zhejiang Shengmei, Hunan Yueneng, Nanjing Blue Solid, BASF Shanshan, Kunlun, Sepion, Mitra Chem, Nascent Materials, Hengchuang, Shanshan Technology (Finlanda), Shan Shan Technology, Hunan TianTai TianRun, Indonesia BTR, You Shan, Stratus Materials | DA | produc catozi, anozi, electroliti, separatori sau litiu |
| Producatori de baterii cu EV | Solid Power, NexTech Batteries, Poly Energy, RJETech (Rongjie), East Saint, QuantumScape, Taibang, CIDEcell, Fengyang Pengen, Rusaccum, Anda Technology, Greenbatt, Zakuro | DA | produc celule de baterii si mentioneaza EV / automotive |
| Baterii pentru EV + incarcare | ReFire_ev | DA | furnizeaza baterii pentru EV (pe langa statii de incarcare) |
| Producatori de baterii fara EV | Wanbang, GuTai, Nuvvon, Xinliyuan | LA LIMITA | produc baterii, dar mentioneaza doar stocare de energie |
| Activitati mixte | RJETech (Youngy) | LA LIMITA | litiu, catozi, anozi (DA), dar se concentreaza pe stocare, nu pe EV |
| Activitati mixte | Shenzhen Atsm Energy | LA LIMITA | materiale pentru baterii (DA) + echipamente pentru fabrici (NU) |
| Doar tehnologie | Charge CCCV | NU | firma de proprietate intelectuala, nu produce si nu furnizeaza nimic fizic |
| Solar / stocare pentru case si proiecte | Quanta Services, Naust Marine, Texon, Sowin Power, CIRKEL Energi (x2, dublura), JN-Solar, Sunnytek, Rhino Power, VASOL TECH, Krefter, Ecopwr, NextGEN Energy, Effecta, Alotta, Magnora, Againity, BRF Vivstavarvsvagen, Folksol | NU | instaleaza sau folosesc baterii, nu produc componente pentru baterii EV |

Total: 43 DA - 6 LA LIMITA - 20 NU (din 69 de candidati)

Observatii:
- cererea e despre rolul in lantul de aprovizionare: conteaza ce produce firma si cui ii vinde, nu doar daca apare cuvantul "battery"
- 20 de firme au "battery" in descriere, dar NU sunt raspunsuri bune (instaleaza panouri solare cu baterii) -> o cautare dupa cuvinte le-ar prinde pe toate (similaritate diferita de relevanta)
- CIRKEL Energi apare de 2 ori -> inca o dublura in date (ca Rompetrol)
- spre deosebire de cererea 6, aici NAICS nu ajuta mult: materialele sunt in "Other Basic Inorganic Chemical Manufacturing", nu in "Battery Manufacturing"